In [30]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

In [31]:
class Value:
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
        
    def __repr__(self):
        return f"Value(data={self.data})"
        
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward  
        
        return out
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"    # giving an assertion error in case the other is neither int nor float
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other -1)) * out.grad
        out._backward = _backward

        return out

    def __rmul__(self, other):  # for the case when we are multiplying a number to a Value node instead of a Value node to a number
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1

    def __sub__(self, other):  # 1, x -> 1 - x
        return self + (-other)

    def __radd__(self, other):  # similar reasoning to rmul
        return self + other

    def __rsub__(self, other):  # x, 1
        return other + (-self)  # 1 - x

    def __rtruediv__(self, other):
        return other * self**-1
        
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')      # creating the value node of the output

        def _backward():                     # defining the backward function for the tanh to call during the back propogation     
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward    # the backward function is stored in the new output value of that is created from the operation

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad 
        out._backward = _backward

        return out

    def relu(self):
        x = self.data
        out = Value(max(0, x), (self, ), 'relu')

        def _backward():
            self.grad += (1.0 if x > 0 else 0.0) * (out.grad)  # i need to check the logic of this out.grad again  -> there is always (local_derivative * incoming_derivative) because we need to find how the output depends on that operation so we have to multiply it with the gradient of its output so we can chain it all together. 
        out._backward = _backward

        return out
        
    def log(self):
        x = self.data
        out = Value(math.log(x), (self, ), 'log')

        def _backward():
            self.grad += (1/x)*out.grad
        out._backward = _backward
        
        return out

    def backward(self):       # defining the backward function that the user will call

        topo = []
        visited = set()      # a set to keep track of the nodes that have been visited

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)   # calling the build_topo function on the child nodes to recursively add all the nodes in the visited set
                topo.append(v)
        build_topo(self)

        self.grad = 1.0    # setting the self.grad to 1 since the grad of the thing to itself is 1
        for node in reversed(topo):    # iterating through the set from the last node to the first node
            node._backward()   # calling the _backward function that is defined for the particular node

def softmax(logits):
    m = max(l.data for l in logits)
    exps = [(l - m).exp() for l in logits]    # we subtract maximum and then exponentiate to prevent overflow this does not chagne the final answer as this gets cancellled out  ->  exp(xi - m) / sumj exp(xj - m) = exp(xi)·exp(-m) / (exp(-m) · sumj exp(xj)) = exp(xi) / sumj exp(xj)
    total = sum(exps)
    return [e / total for e in exps]

def cross_entropy(logits, target):  # target is the correct answer for that example as an integer index it tells which class the example really belongs to -> it is there in the dataset labels
    return -softmax(logits)[target].log()     # we know log1 = 0, so as close the value of the probabilility is of the coorect class to 1 then lower the cross entropy loss will be.
                                              # we take the negative log lokliehood of the answer because we want to use gradient descent algo to reduce the loss, and grad descent works by decreasing the value of the loss from high to 0 instead of increasing a negative number to 0 and that is why we take the negative to make it work.

# only that _backward function is getting defined for the self on whom the particular operation 
# has been performed the other _backward function in other operator defination is not getting defined.

# lambda: None is to make sure that even when no _backward function is defined (for the case when
# no operation is performed to get that value) then that lambda: None will run making sure that the code does not crash.

# we have to add the values in the grads of the nodes so we use += in case that the same node is called multiple times

In [32]:
class Neuron:

    def __init__(self, nin):    # defining random random numbers i.e. weights and biases (both between -1 and 1) for the nin number of inputs each 
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        # w * x + b
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()   # activation function is called on each of the neuron in the list
        return out    # whenever we call the Neuron object with the inputs x(list) then this will be called and the value will be returned

    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):   # this will create the layer of nout number of neurons for the nin number of input neurons where each of the nin neurons is connected to the every single nout number of neurons of layer and all these nout number of neurons will be added to the list "neurons"
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]   # activation function is called for each of the neuron in each of the list of neurons in the neurons list
        return outs[0] if len(outs) == 1 else outs   # for making the output look cleaner in case only 1 output is there

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:

    def __init__(self, nin, nout):
        sz = [nin] + nout
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nout))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    

In [33]:
n = MLP(3, [4, 4, 1])

In [34]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [35]:
for k in range(20):
  
  # forward pass
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  
  # backward pass
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  
  # update
  for p in n.parameters():
    p.data += -0.1 * p.grad
  
  print(k, loss.data)

0 3.935896662840074
1 3.1586174676141154
2 2.985820289922987
3 2.9007652987148576
4 2.8486141937433445
5 2.812783253702589
6 2.786274118943808
7 2.765550762398309
8 2.748590938368645
9 2.7341328044305047
10 2.721312543032674
11 2.7094866856815525
12 2.6981172650939276
13 2.6866975198496466
14 2.674676755233901
15 2.66137759952497
16 2.6458637468885637
17 2.6267203765065092
18 2.601620636626839
19 2.566436145172776


In [36]:
ypred

[Value(data=0.3489758275303657),
 Value(data=-0.8970288160156191),
 Value(data=0.28628878775425465),
 Value(data=0.30901392069047545)]

In [37]:
x = Value(3.0)
print(1-x)
print(x-1)

Value(data=-2.0)
Value(data=2.0)


In [49]:
logits = [Value(2.0), Value(1.0), Value(0.1)]
loss = cross_entropy(logits, 0)
loss.backward()
print(loss.data)
print([l.grad for l in logits])

0.4170300162778335
[-0.3409988611140321, 0.24243297070471392, 0.09856589040931818]


In [51]:
logits = [Value(2.0), Value(1.0), Value(0.1)]
loss = cross_entropy(logits, 2)
loss.backward()
print(loss.data)
print([l.grad for l in logits])    # negative gradient => that if you increase this logit then the loss will decrease

2.3170300162778332
[0.6590011388859679, 0.24243297070471392, -0.9014341095906818]


In [62]:
x = Value(3.0)
print(1 - x)
print(x - 1)

Value(data=-2.0)
Value(data=2.0)
